# Домашняя работа 3

## Подготовка

In [6]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.2 (6830, 7)


## Исходники домашней работы

In [7]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

                    lei                       organization_name  source_count  \
0  549300AG64NHILB7ZP05                      LOANDEPOT.COM, LLC           114   
1  QFROUN1UWUYU0DVIWD51  Fifth Third Bank, National Association             3   
2  5493003GQDUH26DNNH17               NAVY FEDERAL CREDIT UNION           742   

   activity_year  
0           2023  
1           2023  
2           2023  


## H1 Профиль отказов

In [8]:
df_h1 = decisions.copy()
df_h1["is_denied"] = df_h1["action_taken"] == 3

denial = (
    df_h1.groupby(org_keys, as_index=False)
    .agg(
        n_decisions=("is_denied", "size"),
        n_denied=("is_denied", "sum"),
    )
)
denial["denial_rate"] = denial["n_denied"] / denial["n_decisions"]


denial = pd.merge(
    denial,
    lookup,
    on=org_keys,
    how="left",
    validate="one_to_one",
    indicator=True,
)


assert (denial["_merge"] == "both").all(), "Найдены организации без названия в lookup"
denial = denial.drop(columns=["_merge"])

assert denial["n_decisions"].sum() == len(df_h1), "Не сходится общее число решений"
assert denial["n_denied"].sum() == df_h1["is_denied"].sum(), "Не сходится общее число отказов"
assert denial["denial_rate"].between(0, 1).all(), "Доля отказов вне диапазона [0, 1]"


denial.to_csv(WORK / "home_denial_profile.csv", index=False)

print("Форма denial:", denial.shape)
print("Пример первой организации для отчёта:")
print(denial.iloc[0][["lei", "activity_year", "organization_name", "n_denied", "n_decisions", "denial_rate"]])


Форма denial: (314, 6)
Пример первой организации для отчёта:
lei                           03D0JEWFDFUS0SEEKG89
activity_year                                 2023
organization_name    TD Bank, National Association
n_denied                                        15
n_decisions                                     91
denial_rate                               0.164835
Name: 0, dtype: object


## H2 Группы для просмотра

In [9]:

large_groups = denial[denial["n_decisions"].ge(20)].copy()
large_groups.to_csv(WORK / "home_denial_groups20.csv", index=False)

total_decisions = denial["n_decisions"].sum()
total_denied = denial["n_denied"].sum()
overall_denial_rate = total_denied / total_decisions
unweighted_mean_rate = denial["denial_rate"].mean()

print(f"Групп всего (полный профиль): {len(denial)}")
print(f"Групп с >= 20 решениями: {len(large_groups)}")
print(f"Общая доля отказов (взвешенная): {overall_denial_rate:.4f} ({total_denied} / {total_decisions})")
print(f"Простое среднее процентов по группам: {unweighted_mean_rate:.4f}")


Групп всего (полный профиль): 314
Групп с >= 20 решениями: 61
Общая доля отказов (взвешенная): 0.1012 (691 / 6830)
Простое среднее процентов по группам: 0.1422


## H3 Длинная, широкая и снова длинная форма

In [11]:

long_base = (
    decisions.groupby(long_keys, as_index=False)
    .size()
    .rename(columns={"size": "n"})
)


wide = (
    long_base.pivot(index=org_keys, columns="action_taken", values="n")
    .reindex(columns=[1, 2, 3])
    .fillna(0)
)


long_restored = (
    wide.reset_index()
    .melt(id_vars=org_keys, value_vars=[1, 2, 3], var_name="action_taken", value_name="n")
)
long_restored["action_taken"] = long_restored["action_taken"].astype("int64")
long_restored["n"] = long_restored["n"].astype("int64")
long_restored = long_restored[long_restored["n"] > 0].copy()


left_cmp = long_base.sort_values(long_keys).reset_index(drop=True)
right_cmp = long_restored[long_keys + ["n"]].sort_values(long_keys).reset_index(drop=True)


assert_frame_equal(left_cmp, right_cmp)
print("assert_frame_equal пройден успешно! Строк:", len(right_cmp))
print("Размер wide-таблицы (строк с нулями):", wide.shape, "| Кол-во искусственных нулей:", (wide == 0).sum().sum())


assert_frame_equal пройден успешно! Строк: 500
Размер wide-таблицы (строк с нулями): (314, 3) | Кол-во искусственных нулей: 442


## H+ Безопасное соединение

In [12]:

lookup_full = lookup.copy()
lookup_partial = lookup.iloc[1:].copy()  
lookup_dup = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)  

lookup_conflict = lookup.copy()
conflict_row = lookup_conflict.iloc[[0]].copy()
conflict_row["organization_name"] = conflict_row["organization_name"] + " (ALT)"
lookup_conflict = pd.concat([lookup_conflict, conflict_row], ignore_index=True)


conflict_counts = lookup_conflict.groupby(org_keys)["organization_name"].nunique(dropna=False)
print("Ключи с >1 уникальным именем в lookup_conflict:", (conflict_counts > 1).sum())

def safe_enrich(left: pd.DataFrame, right: pd.DataFrame, keys: list[str], value_col: str = "organization_name"):
    missing_left = [k for k in keys if k not in left.columns]
    missing_right = [k for k in keys if k not in right.columns]
    if missing_left or missing_right:
        raise KeyError(f"Отсутствуют столбцы ключей. Слева: {missing_left}, справа: {missing_right}")
    
    if right.duplicated(subset=keys).any():
        dup_keys = right.loc[right.duplicated(subset=keys, keep=False), keys].drop_duplicates()
        raise ValueError(f"Обнаружены повторные ключи справа ({len(dup_keys)} шт.) — риск дублирования строк при JOIN")
    
    merged = pd.merge(
        left,
        right[keys + [value_col]],
        on=keys,
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    assert len(merged) == len(left), "Нарушена кардинальность исходной таблицы"
    
    left_only_report = merged.loc[merged["_merge"] == "left_only", keys].copy()
    result = merged.drop(columns=["_merge"])
    return result, left_only_report

base_denial = denial[org_keys + ["n_decisions", "n_denied", "denial_rate"]].copy()

res_full, lo_full = safe_enrich(base_denial, lookup_full, org_keys)
print(f"[1. Full] Строк: {len(res_full)}, left_only: {len(lo_full)}")


res_part, lo_part = safe_enrich(base_denial, lookup_partial, org_keys)
print(f"[2. Partial] Строк: {len(res_part)}, left_only: {len(lo_part)} (ключ: {lo_part.to_dict('records')})")

try:
    safe_enrich(base_denial, lookup_dup, org_keys)
except ValueError as e:
    print(f"[3. Duplicate] Ожидаемая ошибка перехвачена: {e}")

try:
    safe_enrich(base_denial, lookup_conflict, org_keys)
except ValueError as e:
    print(f"[4. Conflict] Ожидаемая ошибка перехвачена: {e}")

try:
    safe_enrich(base_denial, lookup_full, org_keys + ["unknown_col"])
except KeyError as e:
    print(f"[5. Unknown key] Ожидаемая ошибка перехвачена: {e}")


Ключи с >1 уникальным именем в lookup_conflict: 1
[1. Full] Строк: 314, left_only: 0
[2. Partial] Строк: 314, left_only: 1 (ключ: [{'lei': '549300AG64NHILB7ZP05', 'activity_year': 2023}])
[3. Duplicate] Ожидаемая ошибка перехвачена: Обнаружены повторные ключи справа (1 шт.) — риск дублирования строк при JOIN
[4. Conflict] Ожидаемая ошибка перехвачена: Обнаружены повторные ключи справа (1 шт.) — риск дублирования строк при JOIN
[5. Unknown key] Ожидаемая ошибка перехвачена: "Отсутствуют столбцы ключей. Слева: ['unknown_col'], справа: ['unknown_col']"
